In [0]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/aemo_raw/")
)

df.printSchema()
print(f"Rows: {df.count():,}")
display(df)

In [0]:
from pyspark.sql import functions as F

prices = (
    df.select(
        F.col("REGION").alias("region_code"),
        F.coalesce(
            F.to_timestamp(F.col("SETTLEMENTDATE").cast("string"), "yyyy/MM/dd HH:mm:ss"),
            F.col("SETTLEMENTDATE").cast("timestamp"),
        ).alias("interval_end_at"),
        F.col("RRP").cast("double").alias("price_aud_per_mwh"),
        F.col("TOTALDEMAND").cast("double").alias("demand_mw"),
    )
    .withColumn("interval_start_at", F.col("interval_end_at") - F.expr("INTERVAL 5 MINUTES"))
    .withColumn("interval_date", F.to_date("interval_start_at"))
    .withColumn("is_negative_price", F.col("price_aud_per_mwh") < 0)
)

display(prices.limit(10))

In [0]:
print("Rows with no timestamp:", prices.filter(F.col("interval_end_at").isNull()).count())

In [0]:
#Cell 4: price by state
display(
    prices.groupBy("region_code")
    .agg(
        F.round(F.avg("price_aud_per_mwh"), 2).alias("avg_price"),
        F.min("price_aud_per_mwh").alias("min_price"),
        F.max("price_aud_per_mwh").alias("max_price"),
    )
    .orderBy(F.desc("avg_price"))
)

In [0]:
#the 10 most expensive intervals

display(
    prices.select("region_code", "interval_end_at", "price_aud_per_mwh")
    .orderBy(F.desc("price_aud_per_mwh"))
    .limit(10)
)

In [0]:
#negative prices by state

display(
    prices.groupBy("region_code")
    .agg(
        F.sum(F.col("is_negative_price").cast("int")).alias("negative_intervals"),
        F.count("*").alias("total_intervals"),
    )
    .withColumn("pct_negative", F.round(100 * F.col("negative_intervals") / F.col("total_intervals"), 1))
    .orderBy(F.desc("pct_negative"))
)

In [0]:
#the same thing in Spark SQL. Spark speaks SQL too:

prices.createOrReplaceTempView("prices")

display(spark.sql("""
    SELECT region_code,
           ROUND(AVG(price_aud_per_mwh), 2) AS avg_price
    FROM prices
    GROUP BY region_code
    ORDER BY avg_price DESC
"""))